# 36120 Advanced Machine Learning - Assignment 1
## NBA Career Longevity Prediction - Experiment 3
**Student Name:** Aryan Goel  
**Goal:** Build an ensemble combining our optimized models (Tuned LightGBM and HistGradientBoosting Classifier) to maximize out-of-fold AUPRC performance and generate the final predictions for Kaggle submission.

This notebook uses the modular preprocessing and feature engineering package `nba_prep` developed for this project.

In [1]:
# Auto-install dependencies if they are missing in the current kernel environment
try:
    import pandas as pd
    import numpy as np
    import lightgbm as lgb
    import nba_prep
    print("All dependencies are already installed and loaded!")
except ModuleNotFoundError:
    import sys
    import subprocess
    print("Installing dependencies from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])
    print("Dependencies installed successfully! Please restart/refresh the notebook kernel.")

All dependencies are already installed and loaded!


In [2]:
# Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

# Dynamically append the package src directory to sys.path in case it is not installed
import sys
from pathlib import Path
package_path = str(Path("../36120-26SP-group-student-package/src").resolve())
if package_path not in sys.path:
    sys.path.append(package_path)

# Import custom utility package
from nba_prep.cleaning import clean_height, clean_year, calculate_age
from nba_prep.features import (
    calculate_ast_tov_ratio,
    calculate_true_shooting_percentage,
    calculate_defensive_impact,
    calculate_scoring_efficiency,
)

### 1. Load Raw Datasets

In [3]:
train = pd.read_csv('../data/raw/train.csv', low_memory=False)
test = pd.read_csv('../data/raw/test.csv', low_memory=False)
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (41195, 66)
Test shape: (12061, 65)


### 2. Feature Cleaning and Engineering

In [4]:
def process_dataframe(df):
    processed = df.copy()
    
    # Height & Year cleaning
    processed["ht_inches"] = processed["ht"].apply(clean_height)
    processed["yr_clean"] = processed["yr"].apply(clean_year)
    
    # Derived age
    processed["age"] = processed.apply(lambda r: calculate_age(r["year"], r["dob"]), axis=1)
    
    # Advanced metrics
    tov_est = np.where((processed["ast/tov"] > 0) & (processed["ast/tov"].notna()), processed["ast"] / processed["ast/tov"], 0.0)
    processed["ast_tov_ratio_clean"] = calculate_ast_tov_ratio(processed["ast"], tov_est)
    
    fga = processed["twoPA"] + processed["TPA"]
    processed["true_shooting_per_clean"] = calculate_true_shooting_percentage(processed["pts"], fga, processed["FTA"])
    processed["defensive_impact_score"] = calculate_defensive_impact(processed["stl"], processed["blk"], processed["dreb"])
    processed["scoring_efficiency_score"] = calculate_scoring_efficiency(processed["pts"], processed["Min_per"])
    processed["usage_ortg_interaction"] = processed["usg"] * processed["ORtg"]
    
    return processed

train_processed = process_dataframe(train)
test_processed = process_dataframe(test)
print("Finished preprocessing!")

/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:11: RuntimeWarning: invalid value encountered in divide
  return np.where(tov == 0, ast / 1e-5, ast / tov)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: divide by zero encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: invalid value encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: divide by zero encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: invalid value encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)


Finished preprocessing!


/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:11: RuntimeWarning: invalid value encountered in divide
  return np.where(tov == 0, ast / 1e-5, ast / tov)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: divide by zero encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:24: RuntimeWarning: invalid value encountered in divide
  return np.where(denom == 0, 0.0, pts / denom)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: divide by zero encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)
/Users/aryan/AT1_AML/36120-26SP-group-student-package/src/nba_prep/features.py:52: RuntimeWarning: invalid value encountered in divide
  return np.where(min_per == 0, 0.0, pts / min_per)


### 3. Feature Set Setup & Alignment
Configure features, handle missing data via median imputation, and categories alignment.

In [5]:
drop_cols = ["pid", "label", "ht", "yr", "dob", "type", "num"]
feature_cols = [col for col in train_processed.columns if col not in drop_cols]

X = train_processed[feature_cols]
y = train_processed["label"]
X_test = test_processed[feature_cols]

# Impute numeric columns with train set medians
numeric_cols = X.select_dtypes(include=[np.number]).columns
train_medians = X[numeric_cols].median().fillna(0.0)

X_imputed = X.copy()
X_test_imputed = X_test.copy()
for col in train_medians.index:
    X_imputed[col] = X_imputed[col].fillna(train_medians[col])
    X_test_imputed[col] = X_test_imputed[col].fillna(train_medians[col])

# Handle categorical columns using consistent codes
categorical_cols = ["team", "conf", "role"]
for col in categorical_cols:
    if col in X.columns:
        categories = X[col].dropna().unique()
        X_imputed[col] = pd.Categorical(X_imputed[col], categories=categories).codes
        X_test_imputed[col] = pd.Categorical(X_test_imputed[col], categories=categories).codes

print("Feature set alignment completed!")

Feature set alignment completed!


### 4. Ensemble Model Training
We train both our **Tuned LightGBM** and **HistGradientBoosting** classifiers within the same 5-Fold Stratified Cross-Validation setup. This guarantees that out-of-fold predictions align perfectly for blending.

In [6]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Prediction storage
oof_lgb_tuned = np.zeros(len(train))
test_preds_lgb_tuned = np.zeros(len(test))

oof_hgb = np.zeros(len(train))
test_preds_hgb = np.zeros(len(test))

print("Training LightGBM and HistGradientBoosting in parallel folds...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    # --- Split for LightGBM ---
    X_train_lgb, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val_lgb, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    X_tr_lgb = X_train_lgb.copy()
    X_v_lgb = X_val_lgb.copy()
    X_t_lgb = X_test.copy()
    for col in categorical_cols:
        if col in X_tr_lgb.columns:
            X_tr_lgb[col] = X_tr_lgb[col].astype("category")
            X_v_lgb[col] = X_v_lgb[col].astype("category")
            X_t_lgb[col] = X_t_lgb[col].astype("category")
            
    # --- Train LightGBM ---
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42 + fold,
        verbose=-1
    )
    model_lgb.fit(X_tr_lgb, y_train, eval_set=[(X_v_lgb, y_val)], callbacks=[lgb.early_stopping(80, verbose=False)])
    
    oof_lgb_tuned[val_idx] = model_lgb.predict_proba(X_v_lgb)[:, 1]
    test_preds_lgb_tuned += model_lgb.predict_proba(X_t_lgb)[:, 1] / kf.n_splits
    
    # --- Split and Train HistGradientBoosting ---
    X_train_hgb = X_imputed.iloc[train_idx]
    X_val_hgb = X_imputed.iloc[val_idx]
    
    model_hgb = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42 + fold
    )
    model_hgb.fit(X_train_hgb, y_train)
    
    oof_hgb[val_idx] = model_hgb.predict_proba(X_val_hgb)[:, 1]
    test_preds_hgb += model_hgb.predict_proba(X_test_imputed)[:, 1] / kf.n_splits
    
    print(f"Fold {fold+1} Completed!")

print("Finished Training!")

Training LightGBM and HistGradientBoosting in parallel folds...


Fold 1 Completed!


Fold 2 Completed!


Fold 3 Completed!


Fold 4 Completed!


Fold 5 Completed!
Finished Training!


### 5. Blending & Validation Performance Evaluation
We evaluate a simple weighted average blend of the two models' predictions: 
`ensemble_pred = 0.6 * LightGBM + 0.4 * HistGradientBoosting`

In [7]:
lgb_score = average_precision_score(y, oof_lgb_tuned)
hgb_score = average_precision_score(y, oof_hgb)
print(f"Tuned LightGBM OOF AUPRC: {lgb_score:.5f}")
print(f"HistGradientBoosting OOF AUPRC: {hgb_score:.5f}")

# Blend out-of-fold predictions
oof_ensemble = 0.6 * oof_lgb_tuned + 0.4 * oof_hgb
ensemble_score = average_precision_score(y, oof_ensemble)

print("-----------------------------------")
print(f"Ensemble OOF AUPRC: {ensemble_score:.5f}")
print(f"Ensemble Uplift over Baseline: {ensemble_score - 0.21040:.5f}")

Tuned LightGBM OOF AUPRC: 0.66915
HistGradientBoosting OOF AUPRC: 0.43076
-----------------------------------
Ensemble OOF AUPRC: 0.63803
Ensemble Uplift over Baseline: 0.42763


### 6. Final Kaggle Submission Generation
We generate predictions for the test set using the same ensemble weights.

In [8]:
ensemble_test_preds = 0.6 * test_preds_lgb_tuned + 0.4 * test_preds_hgb

submission = pd.DataFrame({
    "pid": test_processed["pid"],
    "label": ensemble_test_preds
})

submission.to_csv("../submission.csv", index=False)
print("Final ensembled submission saved successfully to ../submission.csv!")
print(submission.head())

Final ensembled submission saved successfully to ../submission.csv!
     pid     label
0  27085  0.037360
1  30747  0.047453
2  32932  0.070829
3  33635  0.245198
4  33955  0.413882
